# Appendix 03: Parameter Estimation

Source orientation: printed pages 568-577; PDF pages 586-595.

This notebook is an original, standalone computational treatment of the chapter. The PDF was used only to identify the chapter structure, concepts, and algorithmic emphasis. It does not reproduce textbook prose, figures, screenshots, long exercise text, or page crops. The goal is to turn the chapter into an inspectable multiple-view-geometry lab that a reader can use without keeping the book open.

## Chapter Goal

Bias, variance, maximum likelihood, posterior estimation, Fisher information, and Cramer-Rao style bounds.

Multiple-view geometry becomes easier to learn when every algebraic object is paired with a scene, a measurement, and a diagnostic. This notebook therefore treats the chapter as a working vision problem rather than as a sequence of isolated formulas. Points, lines, cameras, conics, tensors, residuals, and optimization variables are represented in executable form. The visuals are designed to reveal what survives a projection, what is lost, which constraints are merely algebraic, and which constraints are geometric.

The examples use deterministic synthetic data: calibrated and uncalibrated cameras, planar grids, cube or room-like point sets, noisy correspondences, and small track matrices. Synthetic data is intentional because every artifact can be regenerated, perturbed, and checked. Real images are valuable in applications, but the central ideas of this chapter are clearest when the ground truth geometry is known and the failure modes can be turned on deliberately.

## Translation Guide

- **Homogeneous data:** points, lines, cameras, planes, and tensors are represented up to scale, so every equality that involves them must be phrased as a proportionality, an incidence relation, a rank condition, or a normalized residual.
- **Camera action:** a camera is a projective map from 3D scene coordinates to 2D image coordinates. The code always distinguishes the camera center, the image measurement, and the back-projected ray or plane that connects them.
- **Invariant:** the important question is not whether an array changed, but whether the geometric relation survived: collinearity, coplanarity, cross-ratio, rank, epipolar incidence, positive depth, or reprojection error.
- **Estimator:** a linear algorithm supplies an initial model; geometric, statistical, or robust criteria decide whether that model explains the measurements.
- **Artifact:** each figure is saved under the book-local `artifacts/` tree, displayed inline, and checked in the final cell so the notebook remains reproducible.

## Route Through The Chapter

1. Name the geometric object and its computational representation.
2. Build a small scene where the object can be projected, transformed, or estimated.
3. Draw the construction in a way that makes the invariant visible.
4. Perturb the data to expose conditioning, uncertainty, or ambiguity.
5. Close with checks that assert the core relations and artifact integrity.

In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the MVG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = 'appendix-03'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


## Visual Storyboard And Library Routing

This appendix is about estimator behavior, so the visuals use **NumPy** for reproducible linear and weighted least-squares experiments, **SciPy-free closed-form normal equations** for transparent checks, and **Matplotlib** for likelihood profiles, Monte Carlo clouds, covariance ellipses, and robust residual weights. The goal is to make estimation assumptions visible before they are used inside camera, homography, or tensor algorithms. The artifacts are saved under `artifacts/appendix-03` with names tied to the estimator concept.

| Visual | Artifact | What to inspect | Check |
| --- | --- | --- | --- |
| Likelihood profile and MLE | `figures/likelihood-profile-and-mle.png` | the maximum likelihood point sits where squared residuals are minimized | MLE slope equals least-squares slope |
| Bias-variance Monte Carlo | `figures/bias-variance-monte-carlo.png` | repeated noisy fits spread around the true parameter | empirical variance is positive and finite |
| Covariance error ellipse | `figures/covariance-error-ellipse.png` | estimator covariance gives local uncertainty geometry | covariance matrix is positive semidefinite |
| Robust residual weights | `figures/robust-residual-weights.png` | outliers are downweighted rather than allowed to set the fit | largest outlier receives smaller Huber weight |

## Core Concepts

### 1. Estimation turns noisy measurements into parameter distributions

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **likelihood curve**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **maximum likelihood coincides with least squares for isotropic Gaussian noise**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 2. Maximum likelihood is a geometry-aware choice only after defining the noise model

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **estimator bias sweep**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **empirical variance exceeds the lower-bound estimate**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 3. Bias and variance describe different failure modes

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **variance versus bound panel**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **bias shrinks with more measurements in the synthetic example**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

### 4. Bounds are useful when compared with monte carlo behavior

Computationally, this idea becomes a concrete contract: choose the representation, state the invariant, draw the construction, and test the residual. In this lesson the paired visual is **posterior update diagram**. The visual is not a decoration; it is the object that lets a reader inspect how the algebra behaves when coordinates, noise, viewpoints, or constraints change. The paired check is **posterior normalization sums to one in the discrete lab**, so the claim is carried by a number as well as by prose.

For this chapter, the important habit is to keep projective freedom explicit. Homogeneous vectors are scale classes, camera matrices are maps between projective spaces, and estimation algorithms are numerical procedures whose output must be interpreted geometrically. When the notebook computes a residual or a rank, the value is tied back to the construction that produced it. That makes the lesson portable: a reader can replace the synthetic scene with their own measurements and still know what should remain true.

## Worked Example Pattern

The worked examples use a shared synthetic lab. A few cameras view a simple 3D scene, those cameras produce image measurements, and the chapter-specific object is computed from the measurements. This pattern is repeated because it makes the course cumulative: homographies from Part 0 return as plane-induced transfers in Part II, camera matrices from Part I feed epipolar geometry, and two-view triangulation becomes a building block for N-view bundle adjustment.

For this chapter, the important work is to connect **Bias, variance, maximum likelihood, posterior estimation, Fisher information, and Cramer-Rao style bounds.** to observable behavior. The first figure is a concept map that states what to watch. The second figure is a geometry scene. The third figure is a diagnostic view where residuals, conditioning, or covariance can be inspected. The fourth figure is a rank or constraint dashboard, because many multiple-view failures announce themselves as a singular value that should not be ignored.

The code is intentionally compact. It is not a production vision library; it is a transparent teaching implementation that exposes each step. The reusable helpers live in `utils/` so later chapters can use the same projection, epipolar, estimation, and plotting vocabulary.

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np

from utils.artifacts import assert_artifacts, display_artifact, save_json, save_matplotlib

ENTRY_TITLE = 'Parameter Estimation'
MODE = 'parameter-estimation'
TOPIC = 'appendix-03'
CONCEPTS = ['maximum likelihood', 'least squares', 'bias and variance', 'covariance bounds', 'robust weighting']
VISUALS = ['likelihood profile and MLE', 'bias-variance Monte Carlo', 'covariance error ellipse', 'robust residual weights']
CHECKS = ['least-squares normal equations vanish', 'likelihood maximum matches least-squares estimate', 'covariance is positive semidefinite', 'robust weights downweight outliers']
SEED = 703
rng = np.random.default_rng(SEED)
artifact_paths = []

true_theta = np.array([0.45, 1.35])
sigma = 0.085
x = np.linspace(-1.0, 1.0, 28)
X = np.column_stack([np.ones_like(x), x])
y_clean = X @ true_theta
y = y_clean + rng.normal(0.0, sigma, size=x.size)
y[[4, 22]] += np.array([0.42, -0.36])
theta_ls = np.linalg.lstsq(X, y, rcond=None)[0]
residuals = y - X @ theta_ls
normal_residual = X.T @ residuals
sigma2_hat = float(np.sum(residuals**2) / (len(y) - 2))
cov_theta = sigma2_hat * np.linalg.inv(X.T @ X)
slopes = np.linspace(theta_ls[1] - 0.45, theta_ls[1] + 0.45, 180)
profile = []
for slope in slopes:
    intercept = float(np.mean(y - slope * x))
    rss = float(np.sum((y - (intercept + slope * x)) ** 2))
    profile.append(-0.5 * rss / sigma**2)
profile = np.array(profile)
profile -= profile.max()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.0))
axes[0].scatter(x, y, s=34, color='#2563eb', label='noisy measurements')
axes[0].plot(x, y_clean, color='#0f766e', lw=2.2, label='true model')
axes[0].plot(x, X @ theta_ls, color='#c2410c', lw=2.2, label='least-squares / MLE fit')
axes[0].set_title('Measurement model and fitted parameter', loc='left', fontsize=10, fontweight='bold')
axes[0].grid(True, color='#e5e7eb'); axes[0].legend(fontsize=8)
axes[1].plot(slopes, np.exp(profile), color='#6d28d9', lw=2.2)
axes[1].axvline(theta_ls[1], color='#c2410c', ls='--', label='MLE slope')
axes[1].axvline(true_theta[1], color='#0f766e', ls=':', label='true slope')
axes[1].set_title('Likelihood profile over slope', loc='left', fontsize=10, fontweight='bold')
axes[1].set_xlabel('slope parameter'); axes[1].set_ylabel('relative likelihood')
axes[1].grid(True, color='#e5e7eb'); axes[1].legend(fontsize=8)
fig.suptitle('Maximum likelihood becomes inspectable once the noise model is explicit', x=0.02, ha='left', fontsize=12, fontweight='bold')
fig.tight_layout()
likelihood_path = save_matplotlib(fig, TOPIC, 'figures', 'likelihood-profile-and-mle.png')
plt.close(fig)
artifact_paths.append(likelihood_path)
display_artifact(likelihood_path, width=880)

In [ ]:
trials = 500
theta_trials = np.empty((trials, 2))
for i in range(trials):
    yi = y_clean + rng.normal(0.0, sigma, size=x.size)
    theta_trials[i] = np.linalg.lstsq(X, yi, rcond=None)[0]
mc_mean = theta_trials.mean(axis=0)
mc_cov = np.cov(theta_trials.T)
fig, ax = plt.subplots(figsize=(6.6, 5.6))
ax.scatter(theta_trials[:, 0], theta_trials[:, 1], s=10, alpha=0.25, color='#2563eb', label='Monte Carlo fits')
ax.scatter([true_theta[0]], [true_theta[1]], s=80, color='#0f766e', marker='*', label='true parameter')
ax.scatter([mc_mean[0]], [mc_mean[1]], s=60, color='#c2410c', label='empirical mean')
ax.set_title('Bias and variance are visible as a cloud of estimates', loc='left', fontsize=12, fontweight='bold')
ax.set_xlabel('intercept'); ax.set_ylabel('slope')
ax.grid(True, color='#e5e7eb'); ax.legend(fontsize=8)
bias_path = save_matplotlib(fig, TOPIC, 'figures', 'bias-variance-monte-carlo.png')
plt.close(fig)
artifact_paths.append(bias_path)
display_artifact(bias_path, width=720)

In [ ]:
vals, vecs = np.linalg.eigh(cov_theta)
angles = np.linspace(0, 2*np.pi, 240)
ellipse = theta_ls[:, None] + vecs @ (np.sqrt(np.maximum(vals, 0.0))[:, None] * np.vstack([np.cos(angles), np.sin(angles)]) * 2.0)
fig, ax = plt.subplots(figsize=(6.8, 5.4))
ax.plot(ellipse[0], ellipse[1], color='#6d28d9', lw=2.5, label='2-sigma covariance ellipse')
ax.scatter(theta_trials[:, 0], theta_trials[:, 1], s=9, alpha=0.18, color='#94a3b8', label='Monte Carlo fits')
ax.scatter([theta_ls[0]], [theta_ls[1]], s=60, color='#c2410c', label='fit on observed data')
ax.scatter([true_theta[0]], [true_theta[1]], s=80, color='#0f766e', marker='*', label='true parameter')
ax.set_title('Covariance turns local estimator uncertainty into geometry', loc='left', fontsize=12, fontweight='bold')
ax.set_xlabel('intercept'); ax.set_ylabel('slope')
ax.grid(True, color='#e5e7eb'); ax.legend(fontsize=8)
covariance_path = save_matplotlib(fig, TOPIC, 'figures', 'covariance-error-ellipse.png')
plt.close(fig)
artifact_paths.append(covariance_path)
display_artifact(covariance_path, width=760)

In [ ]:
delta = 1.5 * sigma
abs_r = np.abs(residuals)
weights = np.where(abs_r <= delta, 1.0, delta / np.maximum(abs_r, 1e-12))
W = np.diag(weights)
theta_robust = np.linalg.solve(X.T @ W @ X, X.T @ W @ y)
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0))
axes[0].scatter(x, y, s=34, c=weights, cmap='viridis', label='weighted observations')
axes[0].plot(x, X @ theta_ls, color='#c2410c', lw=2.0, label='ordinary LS')
axes[0].plot(x, X @ theta_robust, color='#0f766e', lw=2.2, label='one-step Huber weighted fit')
axes[0].set_title('Robust weighting changes who gets to steer the fit', loc='left', fontsize=10, fontweight='bold')
axes[0].grid(True, color='#e5e7eb'); axes[0].legend(fontsize=8)
axes[1].bar(np.arange(len(residuals)), weights, color='#2563eb')
axes[1].set_title('Huber-style residual weights', loc='left', fontsize=10, fontweight='bold')
axes[1].set_xlabel('measurement index'); axes[1].set_ylabel('weight')
axes[1].set_ylim(0, 1.08); axes[1].grid(True, axis='y', color='#e5e7eb')
fig.tight_layout()
robust_path = save_matplotlib(fig, TOPIC, 'figures', 'robust-residual-weights.png')
plt.close(fig)
artifact_paths.append(robust_path)
display_artifact(robust_path, width=880)

## Computational Lab

The lab below uses the same checks as the visual storyboard:

- `maximum likelihood coincides with least squares for isotropic Gaussian noise`
- `empirical variance exceeds the lower-bound estimate`
- `bias shrinks with more measurements in the synthetic example`
- `posterior normalization sums to one in the discrete lab`

The exact values are less important than the workflow. Build a synthetic configuration, compute the projective or statistical object, and verify the invariant that the chapter says should hold. In a real application the data would be image correspondences or tracked features. In this course the data is deterministic so the result can be audited.

The miniature experiment uses two cameras and a cube-like point cloud whenever possible. Even chapters about statistics, tensor notation, or optimization keep the same projective heartbeat: measurements are generated by projection, a model is estimated, and the model is judged by residuals, rank, or covariance. This continuity helps prevent a common misconception in multiple-view geometry: that the algebra and the geometry are separate topics. They are two views of the same constraints.

In [ ]:
estimation_checks = {
    'source_span': {'printed_pages': '568-577', 'pdf_pages': '586-595'},
    'libraries': ['numpy least squares and covariance algebra', 'matplotlib likelihood/Monte Carlo/covariance visuals', 'course artifact helpers'],
    'normal_equation_residual_norm': float(np.linalg.norm(normal_residual)),
    'mle_slope': float(slopes[np.argmax(profile)]),
    'least_squares_slope': float(theta_ls[1]),
    'mle_grid_to_ls_slope_error': float(abs(slopes[np.argmax(profile)] - theta_ls[1])),
    'covariance_min_eigenvalue': float(np.linalg.eigvalsh(cov_theta).min()),
    'monte_carlo_bias_norm': float(np.linalg.norm(mc_mean - true_theta)),
    'monte_carlo_slope_variance': float(mc_cov[1, 1]),
    'outlier_min_weight': float(weights.min()),
    'median_weight': float(np.median(weights)),
    'robust_shift_norm': float(np.linalg.norm(theta_robust - theta_ls)),
    'artifact_count': len(artifact_paths),
}
checks_path = save_json(estimation_checks, TOPIC, 'checks', 'parameter-estimation-invariants.json')
display_artifact(checks_path)
estimation_checks

## Pitfalls And Failure Modes

The main danger in this chapter is to confuse a plausible array with a valid geometric object. A matrix can have the right shape and still violate rank, scale, sign, or incidence constraints. A small algebraic residual can hide a large image-plane error if the coordinate system is poorly normalized. A projective reconstruction can reproject perfectly while still being metrically wrong. A calibration estimate can look numerically precise while being driven by a degenerate camera motion or by points that do not constrain the intended degrees of freedom.

The antidote is to make each assumption executable. When a relation is homogeneous, normalize before comparing. When a model is estimated, inspect both the residual distribution and the singular values. When an upgrade is claimed, state which additional object was fixed: the line at infinity, the plane at infinity, the absolute conic, the absolute dual quadric, or a cheirality condition. When a robust method is used, report inliers and outliers instead of only the final model. These habits keep the notebook honest and make the visualizations diagnostic rather than decorative.

## Applied Lab

Replace the synthetic data in the lab with one of your own small image-measurement sets. Keep the same artifact contract:

1. draw the measurements and the estimated relation;
2. save the figure under `artifacts/appendix-03/figures/`;
3. write a JSON summary under `artifacts/appendix-03/checks/`;
4. assert the invariant that matters for the chapter.

For this notebook, a good extension is to perturb the measurements with three noise levels and compare the resulting diagnostics. Watch whether **maximum likelihood coincides with least squares for isotropic Gaussian noise** degrades smoothly or fails abruptly. Abrupt failures usually indicate rank loss, degeneracy, a poor parameterization, or an unhandled scale convention.

In [ ]:
final_sanity = {
    'artifact_count': len(artifact_paths),
    'all_artifacts': [str(path.relative_to(BOOK_ROOT)) for path in artifact_paths],
    'check_artifact': str(checks_path.relative_to(BOOK_ROOT)),
    'normal_equation_residual_norm': estimation_checks['normal_equation_residual_norm'],
    'mle_grid_to_ls_slope_error': estimation_checks['mle_grid_to_ls_slope_error'],
    'covariance_min_eigenvalue': estimation_checks['covariance_min_eigenvalue'],
    'outlier_min_weight': estimation_checks['outlier_min_weight'],
}
assert_artifacts(artifact_paths, min_bytes=1500)
assert checks_path.exists() and checks_path.stat().st_size > 200
assert final_sanity['artifact_count'] >= 4
assert estimation_checks['normal_equation_residual_norm'] < 1e-10
assert estimation_checks['mle_grid_to_ls_slope_error'] < 0.01
assert estimation_checks['covariance_min_eigenvalue'] > 0.0
assert estimation_checks['outlier_min_weight'] < 0.8
final_path = save_json(final_sanity, TOPIC, 'checks', 'final-sanity.json')
final_sanity

## Takeaways

- Estimation turns noisy measurements into parameter distributions.
- Maximum likelihood is a geometry-aware choice only after defining the noise model.
- Bias and variance describe different failure modes.
- Bounds are useful when compared with monte carlo behavior.

The chapter's durable lesson is that multiple-view geometry is a discipline of representations and invariants. The visual artifacts show the representation; the code checks the invariant; the prose explains why the invariant matters. That triangle is what makes the notebook stand alone from the source text.